# this is training the CNNPZ model on the noisy mock data with randomly dropping bands

In [ ]:
import sys

from packaging import version
import sklearn
from sklearn.model_selection import KFold, train_test_split

assert version.parse(sklearn.__version__) >= version.parse("1.0.1")

import tensorflow as tf

assert version.parse(tf.__version__) >= version.parse("2.8.0")

#import tensorflow_probability as tfp

import matplotlib.pyplot as plt

plt.rc('font', size=14)
plt.rc('axes', labelsize=14, titlesize=14)
plt.rc('legend', fontsize=14)
plt.rc('xtick', labelsize=10)
plt.rc('ytick', labelsize=10)

import pandas as pd
import h5py
import numpy as np

import json
import os

from matplotlib import colors

In [ ]:
import cnnpz

In [ ]:
# Parametric paths (edit via environment variables, or defaults below)
USER = os.environ.get("USER", "jaimerz")
PSCRATCH = os.environ.get("PSCRATCH", f"/pscratch/sd/{USER[0]}/{USER}")
PROJECT_HOME = os.environ.get("CNNPZ_PROJECT_HOME", f"/global/homes/{USER[0]}/{USER}/UCL")

CARDINAL_DATA_ROOT = os.path.join(PSCRATCH, "cnnpz", "cardinal") + "/"
MODEL_ROOT = os.path.join(PSCRATCH, "cnnpz", "noisy_Cardinal", "models")
PRETRAINING_DATA_ROOT = os.path.join(PSCRATCH, "pop-cosmos-data")
FILTER_ROOT = os.path.join(PROJECT_HOME, "rail_base", "src", "rail", "examples_data", "estimation_data", "data", "FILTER") + "/"


## Load training and test data, filter curves

In [ ]:
saveroot = CARDINAL_DATA_ROOT
fname = saveroot + "train_100k_noisy_y1_i23.parquet"
training_y1 = pd.read_parquet(fname)
training_y1 = training_y1.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = saveroot + "train_100k_noisy_y10_i25.4.parquet"
training_y10 = pd.read_parquet(fname)
training_y10 = training_y10.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

saveroot = CARDINAL_DATA_ROOT
fname = saveroot + "test_100k_noisy_y1_i23.parquet"
test_y1 = pd.read_parquet(fname)
test_y1 = test_y1.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = saveroot + "test_100k_noisy_y10_i25.4.parquet"
test_y10 = pd.read_parquet(fname)
test_y10 = test_y10.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

### Transform data, make X_train, Y_train, and test dataset, variants of this dataset

In [ ]:
# get the LSST and roman filter curves:
filter_root = FILTER_ROOT

wave = {
    "Y":106,
    "J":129,
    "H":158,
}
lsst_filter_curves = {}
roman_filter_curves = {}
for b in "ugrizy":
  lsst_filter_curves[b] = np.loadtxt(filter_root + f'DC2LSST_{b}.res')

for b in "YJH":
  roman_filter_curves[b] = np.loadtxt(filter_root + f'roman_{b}{wave[b]}.res')

# define the wavelength grid
# to begin with, use blocks rather than filter curves
lambda_min = lsst_filter_curves['u'][:,0].min()
lambda_max = roman_filter_curves['H'][:,0].max()
print(lambda_min,lambda_max)

lambda_array = np.linspace(lambda_min,lambda_max,31)
# now let's conver filter curves to blocks: here let's also ignore the big Y band as it overlaps with the y band

In [ ]:
bin_edges = {}
for b in "ugrizyJH":
  if b not in "JH":
    bin_edges[b] = cnnpz.get_bin_edges(lsst_filter_curves[b][:,0])
  else:
    bin_edges[b] = cnnpz.get_bin_edges(roman_filter_curves[b][:,0])
new_edges = lambda_array

rebinned_filters={}
for b in "ugrizyJH":
  if b not in "JH":
    counts = lsst_filter_curves[b][:,1]
  else:
    counts = roman_filter_curves[b][:,1]
  rebinned_filters[b] = cnnpz.rebin_filter(bin_edges[b], counts, new_edges)

lambda_array_cen = (new_edges[1:] + new_edges[:-1])/2
dlambda = new_edges[1] - new_edges[0]
#for b in "ugrizyJH":
#  plt.plot(lambda_array_cen, rebinned_filters[b],'.-')

# further convert this to blocks:
# let's take 1 when the current filter value is > next filter
filter_blocks={}
bands = "ugrizyJH"
for i, b in enumerate("ugrizyJ"):
  if i>0:
    filter_blocks[b] = (rebinned_filters[b] >= rebinned_filters[bands[i+1]]) & (rebinned_filters[b] > rebinned_filters[bands[i-1]])
  else:
    filter_blocks[b] = (rebinned_filters[b] > rebinned_filters[bands[i+1]])
filter_blocks['H'] = (rebinned_filters['H'] > rebinned_filters['J']) & (rebinned_filters['H'] > rebinned_filters['y'])

In [ ]:
X_y1, Y_y1 = cnnpz.transform_data_to_XY(training_y1, lambda_array_cen, filter_blocks, apply_stretch = False)
X_test_y1, Y_test_y1 = cnnpz.transform_data_to_XY(test_y1, lambda_array_cen, filter_blocks, apply_stretch = False)

X_y10, Y_y10 = cnnpz.transform_data_to_XY(training_y10, lambda_array_cen, filter_blocks, apply_stretch = False)
X_test_y10, Y_test_y10 = cnnpz.transform_data_to_XY(test_y10, lambda_array_cen, filter_blocks, apply_stretch = False)

In [ ]:
print(X_y1.shape, Y_y1.shape)

## Making datasets with missing bands - NIR removal

In [ ]:
# let's construct a dataset where half of the sample do not have NIR measurements:
X_y1_misnir, Y_y1_misnir = cnnpz.make_incomplete_nir_data(training_y1, lambda_array_cen, filter_blocks, apply_stretch = False)
X_test_y1_misnir, Y_test_y1_misnir = cnnpz.make_incomplete_nir_data(test_y1, lambda_array_cen, filter_blocks, apply_stretch = False)

X_y10_misnir, Y_y10_misnir = cnnpz.make_incomplete_nir_data(training_y10, lambda_array_cen, filter_blocks, apply_stretch = False)
X_test_y10_misnir, Y_test_y10_misnir = cnnpz.make_incomplete_nir_data(test_y10, lambda_array_cen, filter_blocks, apply_stretch = False)

# visualize the data

In [ ]:
cnnpz.visualize_the_data(X_y1, Y_y1, lambda_array_cen, filter_blocks, title="Y1 training example")

In [ ]:
cnnpz.visualize_the_data(X_y1_misnir, Y_y1_misnir, lambda_array_cen, filter_blocks, title="Y1 misnir example")

In [ ]:
cnnpz.visualize_the_data(X_y10, Y_y10, lambda_array_cen, filter_blocks, title="Y10 training example")

In [ ]:
cnnpz.visualize_the_data(X_y10_misnir, Y_y10_misnir, lambda_array_cen, filter_blocks, title="Y10 misnir example")

# training ensemble model

In [ ]:
# train on Y1 complete (or load if already trained)
from cnnpz import build_model
root = MODEL_ROOT
save_dir = root + "/y1_complete_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y1, Y_y1)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.001))

In [ ]:
# Final prediction on test set
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y1)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y1, y_pred_ensemble.flatten(), test_y1['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# also plot the STD as a function of redshift and i-band magnitude:
fig,axarr=plt.subplots(1,3,figsize=[13,4])
plt.sca(axarr[0])
ind = y_pred_STD.flatten() >= 0.05
plt.scatter(Y_test_y1[~ind], y_pred_ensemble.flatten()[~ind], s=0.1, color='k')
plt.scatter(Y_test_y1[ind], y_pred_ensemble.flatten()[ind], s=0.5, color='r', label="STD_y > 0.05")
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (mean)")
plt.legend()

plt.sca(axarr[1])
plt.scatter(Y_test_y1, y_pred_STD.flatten(), s=0.1, color='k')
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (STD)")

plt.sca(axarr[2])
plt.scatter(test_y1['mag_i_lsst'], y_pred_STD.flatten(), s=0.1, color='k')
plt.xlabel("i magnitude")
plt.ylabel("predicted redshift (STD)")

plt.tight_layout()

In [ ]:
# now let's train on the data with dropped bands (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y1_misnir_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y1_misnir, Y_y1_misnir)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
y_pred_ensemble_misnir, y_pred_STD_misnir = cnnpz.ensemble_predict(trained_models, X_test_y1_misnir)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y1_misnir, y_pred_ensemble_misnir.flatten(), test_y1['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1_misnir, y_pred_ensemble_misnir.flatten(), 
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# let's apply the complete test data on the model trained with incomplete data
y_pred_ensemble2, y_pred_STD2= cnnpz.ensemble_predict(trained_models, X_test_y1)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y1, y_pred_ensemble2.flatten(), test_y1['mag_i_lsst'], save=True, saveroot=save_dir + "/stats_on_complete_test.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1, y_pred_ensemble2.flatten(), 
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

## Cardinal Y10

In [ ]:
# train on Y10 complete (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y10_complete_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y10, Y_y10)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.005))

In [ ]:
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y10)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y10, y_pred_ensemble.flatten(), test_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y10, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# also plot the STD as a function of redshift and i-band magnitude:
fig,axarr=plt.subplots(1,3,figsize=[13,4])
plt.sca(axarr[0])
ind = y_pred_STD.flatten() >= 0.05
plt.scatter(Y_test_y10[~ind], y_pred_ensemble.flatten()[~ind], s=0.1, color='k')
plt.scatter(Y_test_y10[ind], y_pred_ensemble.flatten()[ind], s=0.5, color='r', label="STD_y > 0.05")
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (mean)")
plt.legend()

plt.sca(axarr[1])
plt.scatter(Y_test_y10, y_pred_STD.flatten(), s=0.1, color='k')
plt.xlabel("Truth redshift")
plt.ylabel("predicted redshift (STD)")

plt.sca(axarr[2])
plt.scatter(test_y10['mag_i_lsst'], y_pred_STD.flatten(), s=0.1, color='k')
plt.xlabel("i magnitude")
plt.ylabel("predicted redshift (STD)")

plt.tight_layout()

In [ ]:
# try applying this model to the specsel data: is this consistent?
root = MODEL_ROOT
save_dir = root + "/y10_complete_ensemble_CNN_6layers"
trained_models = cnnpz.load_ensemble(save_dir=save_dir)

In [ ]:
saveroot = CARDINAL_DATA_ROOT
training_y10_spec = pd.read_parquet(saveroot + "train_100k_noisy_y10_i25.4.SpecSelect.parquet")
# change the name of the roman columns:
training_y10_spec = training_y10_spec.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

X_y10_spec, Y_y10_spec = cnnpz.transform_data_to_XY(training_y10_spec, lambda_array_cen, filter_blocks, apply_stretch=False)
y_pred_ensemble_spec, y_pred_STD_spec = cnnpz.ensemble_predict(trained_models, X_y10_spec)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_y10_spec, y_pred_ensemble_spec.flatten(), 
                                                     training_y10_spec['mag_i_lsst'], save=True, saveroot=save_dir + "/stats_on_specsel.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_y10_spec, y_pred_ensemble_spec.flatten(), 
           redshift_bins, imag_bins, training_y10_spec['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

This works well on the spec selected sample!

## dropping nir data

In [ ]:
# train on Y10 with dropped NIR bands (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y10_misnir_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y10_misnir, Y_y10_misnir)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.01))

In [ ]:
y_pred_ensemble_misnir, y_pred_STD_misnir = cnnpz.ensemble_predict(trained_models, X_test_y10_misnir)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y10_misnir, y_pred_ensemble_misnir.flatten(), test_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y10_misnir, y_pred_ensemble_misnir.flatten(), 
           redshift_bins, imag_bins, test_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

## Spec select sample (no pre-training):

In [ ]:
saveroot = CARDINAL_DATA_ROOT
training_y10_spec = pd.read_parquet(saveroot + "train_100k_noisy_y10_i25.4.SpecSelect.parquet")
# change the name of the roman columns:
training_y10_spec = training_y10_spec.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})
training_y10_spec = training_y10_spec.reset_index(drop=True) 
X_y10_spec, Y_y10_spec = cnnpz.transform_data_to_XY(training_y10_spec, lambda_array_cen, filter_blocks, apply_stretch=False)


saveroot = CARDINAL_DATA_ROOT
training_y1_spec = pd.read_parquet(saveroot + "train_100k_noisy_y1_i23.SpecSelect.parquet")
# change the name of the roman columns:
training_y1_spec = training_y1_spec.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})
training_y1_spec = training_y1_spec.reset_index(drop=True) 
X_y1_spec, Y_y1_spec = cnnpz.transform_data_to_XY(training_y1_spec, lambda_array_cen, filter_blocks, apply_stretch=False)

In [ ]:
cnnpz.visualize_the_data(X_y1_spec, Y_y1_spec, lambda_array_cen, filter_blocks, title="Y1 spec example")

In [ ]:
# train on Y1 spec-select (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y1_specsel_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y1_spec, Y_y1_spec)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.002))

In [ ]:
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y1)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y1, y_pred_ensemble.flatten(), test_y1['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y1, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y1['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

### Y10

In [ ]:
# train on Y10 spec-select (or load if already trained)
root = MODEL_ROOT
save_dir = root + "/y10_specsel_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y10_spec, Y_y10_spec)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.002))

In [ ]:
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_test_y10)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y10, y_pred_ensemble.flatten(), test_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y10, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

## Now include pre-training with the pop-cosmos data

In [ ]:
# load pre-training data
fname = os.path.join(PRETRAINING_DATA_ROOT, "mock_catalog_Ch1_26.degrade_lsst_y1.parquet")
pretraining_y1 = pd.read_parquet(fname)
# change the name of the roman columns:
pretraining_y1 = pretraining_y1.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

fname = os.path.join(PRETRAINING_DATA_ROOT, "mock_catalog_Ch1_26.degrade_lsst_y10.parquet")
pretraining_y10 = pd.read_parquet(fname)
# change the name of the roman columns:
pretraining_y10 = pretraining_y10.rename(columns={'Roman_obs_J129': 'mag_J_roman', 
                                              'Roman_obs_H158': 'mag_H_roman'})

In [ ]:
# plot comparison in colour-redshift space:
fig,axarr=plt.subplots(1,3,figsize=[10,3],sharey=True)

for ii, colours in enumerate(['gr','ri','iz']):
    plt.sca(axarr[ii])
    ri = pretraining_y10[f'mag_{colours[0]}_lsst'] - pretraining_y10[f'mag_{colours[1]}_lsst']
    red = pretraining_y10['redshift']
    ind = pretraining_y10['mag_i_lsst'] < 25.4
    plt.scatter(red[ind][::50], ri[ind][::50], s=0.1, label="pre-training data")
    
    ri = training_y10[f'mag_{colours[0]}_lsst'] - training_y10[f'mag_{colours[1]}_lsst']
    red = training_y10['redshift']
    plt.scatter(red[::10], ri[::10], s=0.1, label="test data")
    
    plt.grid()
    plt.legend()
    plt.xlabel("redshift")
    plt.ylabel(f"{colours[0]}-{colours[1]}")
    plt.ylim([-1.5,5])

In [ ]:
# pre-training data:
pretraining_y10 = pretraining_y10[pretraining_y10['redshift']<2.4]
pretraining_y10 = pretraining_y10.sample(frac=0.5)
pretraining_y10 = pretraining_y10.reset_index(drop=True)
X_y10_pre, Y_y10_pre = cnnpz.transform_data_to_XY(pretraining_y10, lambda_array_cen, filter_blocks, apply_stretch=False)

In [ ]:
Y_y10_pre.dtype, Y_y10_spec.dtype

In [ ]:
len(Y_y10_pre)

In [ ]:
cnnpz.visualize_the_data(X_y10_pre, Y_y10_pre, lambda_array_cen, filter_blocks, title="Y10 pre-training example")

In [ ]:
# train on Y10 pre-training data (or load if already trained)
root = "/pscratch/sd/q/qhang/cnnpz/noisy_Cardinal/models"
save_dir = root + "/y10_pretrain_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models = cnnpz.load_ensemble(save_dir=save_dir)
    histories = None
else:
    trained_models, histories = cnnpz.train_ensembles(build_model, X_y10_pre, Y_y10_pre)
    cnnpz.save_ensemble(trained_models, save_dir=save_dir)

In [ ]:
if histories is not None:
    cnnpz.plot_ensemble_losses(histories, ylim=(0, 0.005))

In [ ]:
# check pre-training performance
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models, X_y10_pre)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_y10_pre, y_pred_ensemble.flatten(), 
                                                     pretraining_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_y10_pre, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, pretraining_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')

In [ ]:
# fine-tune the pre-trained model on spec-select data (or load if already fine-tuned)
root = "/pscratch/sd/q/qhang/cnnpz/noisy_Cardinal/models"
save_dir = root + "/y10_finetune_ensemble_CNN_6layers"
if os.path.exists(os.path.join(save_dir, "norm_params.json")):
    trained_models_finetune = cnnpz.load_ensemble(save_dir=save_dir)
    histories_finetune = None
else:
    trained_models_finetune, histories_finetune = cnnpz.fine_tune_pre_trained_model(
        X_y10_spec, Y_y10_spec, pretrained_models=trained_models, model_root="", nlayers_forzen=4)
    cnnpz.save_ensemble(trained_models_finetune, save_dir=save_dir)

In [ ]:
if histories_finetune is not None:
    cnnpz.plot_ensemble_losses(histories_finetune, ylim=(0, 0.05))

In [ ]:
y_pred_ensemble, y_pred_STD = cnnpz.ensemble_predict(trained_models_finetune, X_test_y10)

In [ ]:
redshift_bins = np.linspace(0,2.5,11)
imag_bins = np.linspace(18, 25.5,11)
stats2, redshift_stats2, imag_stats2 = cnnpz.get_all_stats(Y_test_y10, y_pred_ensemble.flatten(), test_y10['mag_i_lsst'], save=True, saveroot=save_dir + "/stats.pkl", 
                                                     redshift_bins = redshift_bins, imag_bins = imag_bins)

cnnpz.plot_stats(stats2, redshift_stats2, imag_stats2, Y_test_y10, y_pred_ensemble.flatten(), 
           redshift_bins, imag_bins, test_y10['mag_i_lsst'], save_path=save_dir + '/photoz_stats.png')